# EfficientNetV2-S · Camera-Aug + SSL (v2)

## Key design choices

| | |
|---|---|
| **Model** | EfficientNetV2-S, ImageNet pretrained |
| **Fine-tuning** | 3-stage progressive unfreeze (head → last 3 blocks → full model) |
| **Class imbalance** | WeightedRandomSampler (inverse-freq) + class-weighted CE loss (√ inverse-freq) |
| **Camera aug** | Per-camera PIL transforms that simulate test-camera appearance; horizontal-flip cams remap `Lateral_lying_left ↔ Lateral_lying_right` |
| **SSL thresholds** | Round 1 = **0.95**, Round 2 = **0.90** |
| **Metric** | Macro F1 |


In [ ]:
# ── Installation ─────────────────────────────────────────────────────────────
!pip install albumentations -q

# ── Kaggle credentials ───────────────────────────────────────────────────────
from google.colab import files
files.upload()   # upload kaggle.json

import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

!kaggle competitions download -c multi-view-pig-posture-recognition
!unzip -q multi-view-pig-posture-recognition.zip
!ls

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import ast, copy, re, time
from collections import Counter

import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, ConcatDataset
from torchvision import transforms, models
import torchvision.transforms.functional as TF

from sklearn.metrics import accuracy_score, f1_score
import matplotlib.pyplot as plt

# ── GPU ───────────────────────────────────────────────────────────────────────
torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    x = torch.randn(10000, 10000, device='cuda')
    t0 = time.time(); _ = x @ x; torch.cuda.synchronize()
    print(f'GPU matmul test: {time.time()-t0:.3f}s')
    del x; torch.cuda.empty_cache()

In [ ]:
# ── Constants ─────────────────────────────────────────────────────────────────
CLASS_NAMES = {
    0: 'Lateral_lying_left',
    1: 'Lateral_lying_right',
    2: 'Sitting',
    3: 'Standing',
    4: 'Sternal_lying',
}
NUM_CLASSES = len(CLASS_NAMES)

BASE_DIR    = Path('multiview_pig_posture_recognition')
TRAIN2_IMGS = BASE_DIR / 'train2_images'
TEST_IMGS   = BASE_DIR / 'test_images'

BATCH_SIZE = 32

# ── 3-stage fine-tuning schedule ─────────────────────────────────────────────
# Stage 1 — frozen backbone, head warmup
EPOCHS_S1 = 5
LR_S1     = 1e-3

# Stage 2 — unfreeze last 3 feature blocks
EPOCHS_S2  = 7
LR_S2      = 1e-4
N_UNFREEZE = 3     # number of feature blocks to unfreeze from the end

# Stage 3 — full model, cosine LR decay
EPOCHS_S3 = 8
LR_S3     = 5e-5

# ── SSL rounds ────────────────────────────────────────────────────────────────
EPOCHS_SSL      = 5
SSL_CONF_THRESH = [0.95, 0.90]   # stricter thresholds
LR_SSL          = [2e-5, 1e-5]

# ── Save paths ────────────────────────────────────────────────────────────────
SAVE_SUPERVISED = 'effnetv2s_supervised.pt'
SAVE_SSL_FINAL  = 'effnetv2s_ssl_final.pt'
SUBMISSION_PATH = 'submission_v2.csv'

print('Constants ✓')
print(f'  Supervised epochs: S1={EPOCHS_S1} + S2={EPOCHS_S2} + S3={EPOCHS_S3} = {EPOCHS_S1+EPOCHS_S2+EPOCHS_S3}')
print(f'  SSL thresholds:    {SSL_CONF_THRESH}')

In [ ]:
# ── Data loading ──────────────────────────────────────────────────────────────
def parse_camera_meta(image_id):
    m = re.match(r'(pen\d+)_(orb|tur)_(cam\d+)_', str(image_id))
    return (m.group(1), m.group(2), m.group(3)) if m else ('unknown', 'unknown', 'unknown')

def add_camera_cols(df):
    df['pen']      = df['image_id'].apply(lambda x: parse_camera_meta(x)[0])
    df['cam_type'] = df['image_id'].apply(lambda x: parse_camera_meta(x)[1])
    df['cam_num']  = df['image_id'].apply(lambda x: parse_camera_meta(x)[2])
    df['camera']   = df['pen'] + '_' + df['cam_type'] + '_' + df['cam_num']
    return df

train2 = pd.read_csv(BASE_DIR / 'train2.csv')
train2['source']      = 'train2'
train2['bbox_parsed'] = train2['bbox'].apply(ast.literal_eval)
train2['class_name']  = train2['class_id'].map(CLASS_NAMES)
train2 = add_camera_cols(train2)

test = pd.read_csv(BASE_DIR / 'test.csv')
test['source']      = 'test'
test['bbox_parsed'] = test['bbox'].apply(ast.literal_eval)
test = add_camera_cols(test)

print(f'Train2: {len(train2):,} instances  |  {train2["image_id"].nunique():,} images')
print(f'Test:   {len(test):,} instances   |  {test["image_id"].nunique():,} images')
print('\nClass distribution (train2):')
for cname, cnt in train2['class_name'].value_counts().items():
    print(f'  {cname:25s}: {cnt:,}')
print('\nCamera distribution (train2):')
print(train2.groupby('camera')['image_id'].count().sort_values(ascending=False).to_string())

In [ ]:
# ── Image utilities ───────────────────────────────────────────────────────────
def load_image(image_id, source):
    folder = {'train2': TRAIN2_IMGS, 'test': TEST_IMGS}[source]
    return Image.open(folder / image_id).convert('RGB')

def crop_with_padding(image, bbox, padding=0.12, make_square=True):
    """Crop pig instance [x, y, w, h] with relative padding, optionally squared."""
    img_w, img_h = image.size
    x, y, w, h   = map(float, bbox)
    x1 = x - w * padding;      y1 = y - h * padding
    x2 = x + w + w * padding;  y2 = y + h + h * padding
    if make_square:
        side   = max(x2 - x1, y2 - y1)
        cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        x1, x2 = cx - side / 2, cx + side / 2
        y1, y2 = cy - side / 2, cy + side / 2
    x1 = max(0, int(round(x1)));      y1 = max(0, int(round(y1)))
    x2 = min(img_w, int(round(x2)));  y2 = min(img_h, int(round(y2)))
    return image.crop((x1, y1, max(x2, x1+1), max(y2, y1+1)))

In [ ]:
# ── Camera-specific augmentations ─────────────────────────────────────────────
#
# Each function transforms a crop from a train camera to approximate the visual
# appearance of its test-camera counterpart (brightness, colour cast, mirror).
#
# IMPORTANT — label remapping on horizontal flip:
#   class 0  Lateral_lying_left  ──►  class 1  Lateral_lying_right
#   class 1  Lateral_lying_right ──►  class 0  Lateral_lying_left
#   classes 2-4 are orientation-invariant → label unchanged

FLIP_LABEL_MAP = {0: 1, 1: 0, 2: 2, 3: 3, 4: 4}


def aug_pen2_tur_cam1(img):
    """hflip + colour cooling."""
    img = TF.hflip(img)
    img = TF.adjust_saturation(img, saturation_factor=0.8)
    img = TF.adjust_brightness(img, brightness_factor=0.95)
    return img


def aug_pen1_tur_cam2(img):
    """hflip + brightness boost + mild desaturation."""
    img = TF.hflip(img)
    img = TF.adjust_brightness(img, brightness_factor=1.35)
    img = TF.adjust_saturation(img, saturation_factor=0.9)
    return img


def aug_pen2_orb_cam1(img):
    """hflip + darken + contrast + greenish shift + partial greyscale blend."""
    img = TF.hflip(img)
    img = TF.adjust_brightness(img, brightness_factor=0.6)
    img = TF.adjust_contrast(img, contrast_factor=1.5)
    arr = np.array(img).astype(float)
    arr[:, :, 0] = (arr[:, :, 0] * 0.85).clip(0, 255)   # R ↓
    arr[:, :, 1] = (arr[:, :, 1] * 1.10).clip(0, 255)   # G ↑
    arr[:, :, 2] = (arr[:, :, 2] * 0.80).clip(0, 255)   # B ↓
    shifted  = Image.fromarray(arr.clip(0, 255).astype(np.uint8))
    grey     = np.array(TF.to_grayscale(shifted, num_output_channels=3)).astype(float)
    blended  = (0.65 * arr + 0.35 * grey).clip(0, 255).astype(np.uint8)
    return Image.fromarray(blended)


def aug_pen2_tur_cam2(img):
    """brightness boost + mild desaturation (no flip)."""
    img = TF.adjust_brightness(img, brightness_factor=1.3)
    img = TF.adjust_saturation(img, saturation_factor=0.85)
    return img


def aug_pen2_orb_cam2(img):
    """darken + contrast + partial greyscale blend (no flip)."""
    img = TF.adjust_brightness(img, brightness_factor=0.60)
    img = TF.adjust_contrast(img, contrast_factor=1.5)
    arr  = np.array(img).astype(float)
    grey = np.array(TF.to_grayscale(img, num_output_channels=3)).astype(float)
    return Image.fromarray((0.65 * arr + 0.35 * grey).clip(0, 255).astype(np.uint8))


# pen1_orb_cam1 / pen1_orb_cam2 — no test-camera counterpart, excluded
CAMERA_AUG_FN = {
    'pen2_tur_cam1': (aug_pen2_tur_cam1, True),   # True  = includes hflip → need label remap
    'pen1_tur_cam2': (aug_pen1_tur_cam2, True),
    'pen2_orb_cam1': (aug_pen2_orb_cam1, True),
    'pen2_tur_cam2': (aug_pen2_tur_cam2, False),  # False = no flip → label unchanged
    'pen2_orb_cam2': (aug_pen2_orb_cam2, False),
}

print('Camera augmentations defined:')
for cam, (_, flip) in CAMERA_AUG_FN.items():
    print(f'  {cam:20s}  hflip={flip}  '
          f'{"→ left↔right label swap" if flip else "→ label unchanged"}')

In [ ]:
# ── Transforms ────────────────────────────────────────────────────────────────
class AddGaussianNoise:
    def __init__(self, std=0.02, p=0.15):
        self.std, self.p = std, p
    def __call__(self, t):
        if torch.rand(1).item() < self.p:
            t = torch.clamp(t + torch.randn_like(t) * self.std, 0., 1.)
        return t

MU  = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# Applied after camera-specific PIL augmentation (supervised training)
BASE_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    AddGaussianNoise(std=0.02, p=0.15),
    transforms.Normalize(MU, STD),
])

# Test inference & pseudo-label confidence scoring — no stochastic noise
VAL_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MU, STD),
])

# Pseudo-labeled test crops during SSL fine-tuning.
# General augmentation only — test images are already in their native domain.
SSL_TRANSFORM = transforms.Compose([
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.05),
    transforms.RandomRotation(degrees=10),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    AddGaussianNoise(std=0.02, p=0.15),
    transforms.Normalize(MU, STD),
])

print('Transforms defined ✓')

In [ ]:
# ── Dataset classes ───────────────────────────────────────────────────────────

class PigDatasetCameraAug(Dataset):
    """
    Supervised training dataset with per-camera augmentation.

    Every instance appears at least once (original crop).
    If its camera has an augmentation registered in CAMERA_AUG_FN a second,
    augmented copy is added.  For cameras that apply a horizontal flip the
    label is remapped via FLIP_LABEL_MAP so that directional postures remain
    consistent:
        Lateral_lying_left (0) <-> Lateral_lying_right (1)
        Sitting / Standing / Sternal_lying (2,3,4) → unchanged
    """
    def __init__(self, df, camera_aug_fn, base_transform, is_train=True):
        self.camera_aug_fn  = camera_aug_fn
        self.base_transform = base_transform
        df = df.reset_index(drop=True)
        self.df = df
        self.samples = []   # (iloc_idx, do_aug, label)
        for idx, row in df.iterrows():
            cam   = row.get('camera', 'unknown')
            label = int(row['class_id'])
            self.samples.append((idx, False, label))
            if is_train and cam in camera_aug_fn:
                _, does_flip = camera_aug_fn[cam]
                aug_label    = FLIP_LABEL_MAP[label] if does_flip else label
                self.samples.append((idx, True, aug_label))

    def __len__(self):  return len(self.samples)

    def __getitem__(self, i):
        row_i, do_aug, label = self.samples[i]
        row  = self.df.iloc[row_i]
        img  = load_image(row['image_id'], row['source'])
        crop = crop_with_padding(img, row['bbox_parsed'])
        if do_aug:
            aug_fn, _ = self.camera_aug_fn[row['camera']]
            crop = aug_fn(crop)
        return self.base_transform(crop), label


class PigTestDataset(Dataset):
    """Returns (tensor, row_id_str) for inference / pseudo-labeling."""
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):  return len(self.df)
    def __getitem__(self, i):
        row  = self.df.iloc[i]
        img  = load_image(row['image_id'], row['source'])
        crop = crop_with_padding(img, row['bbox_parsed'])
        return self.transform(crop), str(row['row_id'])


class PseudoLabelDataset(Dataset):
    """
    High-confidence pseudo-labeled test crops for SSL fine-tuning.
    Required columns: image_id, source, bbox_parsed, class_id.
    Uses SSL_TRANSFORM (general augmentation, no camera-specific transforms
    because test images are already captured by their native cameras).
    """
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):  return len(self.df)
    def __getitem__(self, i):
        row  = self.df.iloc[i]
        img  = load_image(row['image_id'], row['source'])
        crop = crop_with_padding(img, row['bbox_parsed'])
        return self.transform(crop), int(row['class_id'])


print('Dataset classes defined ✓')

In [ ]:
# ── Class-imbalance helpers ───────────────────────────────────────────────────
#
# Two complementary mechanisms for rare-class boosting:
#
#  1. WeightedRandomSampler — inverse-frequency weights ensure every class
#     appears roughly equally often inside each mini-batch.
#
#  2. Class-weighted CrossEntropyLoss — √(inverse-frequency) weights amplify
#     the gradient for rare-class errors without over-correcting.
#     (Sitting is typically the rarest class, so it benefits the most.)
#
# Using both together gives balanced exposure AND boosted gradient signal,
# which is the standard recipe when macro-F1 is the target metric.

def compute_loss_weights(labels):
    """
    √(inverse-frequency) class weights for CrossEntropyLoss.
    Moderate boost: rare classes get higher weight, but not as extreme
    as full inverse-frequency (which can cause instability).
    """
    counts  = np.bincount(np.asarray(labels, dtype=int), minlength=NUM_CLASSES)
    inv_f   = 1.0 / np.maximum(counts, 1)
    weights = np.sqrt(inv_f / inv_f.mean())   # √ inverse-freq, mean-normalised
    return torch.FloatTensor(weights).to(DEVICE)


def build_weighted_loader(dataset, labels, batch_size, num_workers=2):
    """
    DataLoader with full inverse-frequency WeightedRandomSampler.
    """
    labels = np.asarray(labels, dtype=int)
    counts = np.bincount(labels, minlength=NUM_CLASSES)
    w      = (1.0 / np.maximum(counts, 1))[labels]
    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(w), num_samples=len(w), replacement=True
    )
    return DataLoader(dataset, batch_size=batch_size, sampler=sampler,
                      num_workers=num_workers, pin_memory=True)


# ── Build supervised training dataset ────────────────────────────────────────
train_ds     = PigDatasetCameraAug(train2, CAMERA_AUG_FN, BASE_TRANSFORM)
train_labels = [s[2] for s in train_ds.samples]
train_loader = build_weighted_loader(train_ds, train_labels, BATCH_SIZE)

class_weights_ce = compute_loss_weights(train_labels)

counts_aug = np.bincount(train_labels, minlength=NUM_CLASSES)
print(f'Dataset: {len(train2):,} raw → {len(train_ds):,} samples after camera aug')
print(f'Batches: {len(train_loader)}')
print('\nClass distribution after camera aug:')
for cid, (cnt, w) in enumerate(zip(counts_aug, class_weights_ce.cpu())):
    print(f'  [{cid}] {CLASS_NAMES[cid]:25s}: {cnt:>6,}  CE-weight={w:.3f}')

In [ ]:
# ── Model + layer-freezing utilities ─────────────────────────────────────────

def create_model():
    m = models.efficientnet_v2_s(
        weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
    )
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
    return m.to(DEVICE)


def freeze_backbone(model):
    """Stage 1: freeze everything except the classifier head."""
    for p in model.features.parameters():    p.requires_grad = False
    for p in model.classifier.parameters():  p.requires_grad = True


def unfreeze_last_n_blocks(model, n=3):
    """
    Stage 2: freeze all backbone layers, then unfreeze the last *n* feature
    blocks.  EfficientNetV2-S features are indexed 0-7; unfreezing from the
    end gives the model task-specific high-level features while keeping
    low-level edge detectors frozen.
    """
    for p in model.parameters():  p.requires_grad = False
    blocks = list(model.features.children())
    for block in blocks[-n:]:
        for p in block.parameters():  p.requires_grad = True
    for p in model.classifier.parameters():  p.requires_grad = True


def unfreeze_all(model):
    """Stage 3: unfreeze every parameter for full-model fine-tuning."""
    for p in model.parameters():  p.requires_grad = True


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


model = create_model()
print(f'EfficientNetV2-S  |  total params: {sum(p.numel() for p in model.parameters()):,}')
print(f'Feature blocks: {len(list(model.features.children()))}')
print('Block names:', [name for name, _ in model.features.named_children()])

In [ ]:
# ── Training utilities ────────────────────────────────────────────────────────

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, preds_all, targets_all = 0.0, [], []
    n = len(loader)
    for bi, (imgs, labels) in enumerate(loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss  += loss.item() * imgs.size(0)
        preds_all.extend(out.argmax(1).detach().cpu().tolist())
        targets_all.extend(labels.cpu().tolist())
        bar = '█' * int(30*(bi+1)/n) + '░' * (30 - int(30*(bi+1)/n))
        print(f'\r  |{bar}| {100*(bi+1)/n:5.1f}%  loss={loss.item():.4f}',
              end='', flush=True)
    print()
    return (total_loss / len(loader.dataset),
            accuracy_score(targets_all, preds_all),
            f1_score(targets_all, preds_all, average='macro', zero_division=0))


def run_stage(model, loader, criterion, optimizer, scheduler,
              n_epochs, stage_name, history, best_f1, best_state):
    """Generic training loop for one fine-tuning stage."""
    print(f'\n{"─"*65}')
    print(f'{stage_name}  |  trainable params: {count_trainable(model):,}')
    print(f'{"─"*65}')
    for epoch in range(n_epochs):
        t0 = time.time()
        loss, acc, f1 = train_one_epoch(model, loader, optimizer, criterion)
        if scheduler: scheduler.step()
        elapsed = time.time() - t0
        history.append({'stage': stage_name, 'epoch': len(history)+1,
                         'train_loss': loss, 'train_acc': acc, 'train_f1': f1})
        flag = '  ← best' if f1 > best_f1 else ''
        lr   = optimizer.param_groups[0]['lr']
        print(f'  Ep {epoch+1:>2}/{n_epochs} ({elapsed/60:.1f}m)  '
              f'loss={loss:.4f}  acc={acc:.4f}  f1={f1:.4f}  lr={lr:.1e}{flag}',
              flush=True)
        if f1 > best_f1:
            best_f1    = f1
            best_state = copy.deepcopy(model.state_dict())
    return best_f1, best_state


def generate_pseudo_labels(model, test_df, transform):
    """Run inference, return test_df copy with 'class_id' and 'confidence' columns."""
    loader = DataLoader(PigTestDataset(test_df, transform),
                        batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)
    preds_all, confs_all = [], []
    model.eval()
    with torch.no_grad():
        for bi, (imgs, _) in enumerate(loader):
            probs      = F.softmax(model(imgs.to(DEVICE)), dim=1)
            confs, preds = probs.max(dim=1)
            preds_all.extend(preds.cpu().tolist())
            confs_all.extend(confs.cpu().tolist())
            bar = '█' * int(20*(bi+1)/len(loader)) + '░' * (20-int(20*(bi+1)/len(loader)))
            print(f'\r  [pseudo] |{bar}| {bi+1}/{len(loader)}', end='', flush=True)
    print()
    result = test_df.copy().reset_index(drop=True)
    result['class_id']   = preds_all
    result['confidence'] = confs_all
    return result


def plot_history(history_list, vlines=None, title='Training curves', fname=None):
    df = pd.DataFrame(history_list)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for col, ax, color, ylabel in [
        ('train_loss', axes[0], 'steelblue', 'Loss'),
        ('train_f1',   axes[1], 'green',     'Macro F1'),
    ]:
        ax.plot(df['epoch'], df[col], marker='o', markersize=3, color=color)
        if vlines:
            for vx, label in vlines:
                ax.axvline(vx, color='gray', linestyle=':', alpha=0.7, label=label)
            ax.legend(fontsize=8)
        ax.set_xlabel('Global epoch'); ax.set_ylabel(ylabel); ax.grid(alpha=0.3)
    best = df['train_f1'].max()
    axes[1].axhline(best, color='red', linestyle='--', linewidth=1,
                    label=f'best={best:.4f}')
    axes[1].legend()
    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    if fname: plt.savefig(fname, dpi=130, bbox_inches='tight')
    plt.show()
    return best


print('Utilities defined ✓')

## Phase 1 — Supervised Training (3-stage progressive unfreezing)

| Stage | Layers trained | Epochs | LR | Rationale |
|---|---|---|---|---|
| **1** | Head only | 5 | 1e-3 | Warm up the new 5-class output layer without disturbing pretrained backbone features |
| **2** | Last 3 blocks + head | 7 | 1e-4 | Fine-tune high-level task-specific features while keeping early edge detectors frozen |
| **3** | Full model | 8 | 5e-5 → 1e-6 (cosine) | Full-model polish with controlled LR decay to avoid overwriting useful representations |

In [ ]:
# ── Criterion (class-weighted CE for rare-class boost) ────────────────────────
criterion = nn.CrossEntropyLoss(weight=class_weights_ce)

history_sup  = []
best_f1_sup  = 0.0
best_state_sup = None

# ┌──────────────────────────────────────────────────────────────────────────┐
# │  STAGE 1 — head warmup (backbone frozen)                                 │
# └──────────────────────────────────────────────────────────────────────────┘
freeze_backbone(model)
opt_s1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR_S1
)
best_f1_sup, best_state_sup = run_stage(
    model, train_loader, criterion, opt_s1, scheduler=None,
    n_epochs=EPOCHS_S1, stage_name='Stage 1 — head only',
    history=history_sup, best_f1=best_f1_sup, best_state=best_state_sup
)

In [ ]:
# ┌──────────────────────────────────────────────────────────────────────────┐
# │  STAGE 2 — last 3 blocks unfrozen                                        │
# └──────────────────────────────────────────────────────────────────────────┘
unfreeze_last_n_blocks(model, n=N_UNFREEZE)
opt_s2 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR_S2
)
best_f1_sup, best_state_sup = run_stage(
    model, train_loader, criterion, opt_s2, scheduler=None,
    n_epochs=EPOCHS_S2, stage_name='Stage 2 — last 3 blocks',
    history=history_sup, best_f1=best_f1_sup, best_state=best_state_sup
)

In [ ]:
# ┌──────────────────────────────────────────────────────────────────────────┐
# │  STAGE 3 — full model, cosine LR decay                                   │
# └──────────────────────────────────────────────────────────────────────────┘
unfreeze_all(model)
opt_s3  = torch.optim.Adam(model.parameters(), lr=LR_S3)
sched_s3 = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_s3, T_max=EPOCHS_S3, eta_min=1e-6
)
best_f1_sup, best_state_sup = run_stage(
    model, train_loader, criterion, opt_s3, scheduler=sched_s3,
    n_epochs=EPOCHS_S3, stage_name='Stage 3 — full model (cosine)',
    history=history_sup, best_f1=best_f1_sup, best_state=best_state_sup
)

# Save + download best supervised checkpoint (before SSL)
model.load_state_dict(best_state_sup)
torch.save({'model_state_dict': best_state_sup,
            'class_names':      CLASS_NAMES,
            'best_train_f1':    best_f1_sup,
            'phase':            'supervised'}, SAVE_SUPERVISED)
print(f'\nPhase 1 best macro-F1: {best_f1_sup:.4f}')
print(f'Saved → {SAVE_SUPERVISED}')

from google.colab import files
files.download(SAVE_SUPERVISED)
print('Pre-SSL model downloaded ✓')

In [ ]:
# Phase 1 training curves with stage boundaries
s1_end = EPOCHS_S1
s2_end = EPOCHS_S1 + EPOCHS_S2
plot_history(
    history_sup,
    vlines=[(s1_end + 0.5, 'S1→S2'), (s2_end + 0.5, 'S2→S3')],
    title='Phase 1 — Supervised (3-stage fine-tuning)',
    fname='phase1_curves.png'
)

## Phase 2 — Semi-Supervised Learning

Two SSL rounds with **stricter thresholds** than the default:

| Round | Confidence threshold | Epochs | LR |
|---|---|---|---|
| 1 | **0.95** | 5 | 2e-5 |
| 2 | **0.90** | 5 | 1e-5 |

Each round re-generates pseudo-labels from the current model, combines high-confidence test instances  
with the full supervised train2 set, and fine-tunes the full (unfrozen) model.

In [ ]:
# ── Load best Phase 1 checkpoint ─────────────────────────────────────────────
ckpt = torch.load(SAVE_SUPERVISED, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Phase 1 model loaded  (F1={ckpt["best_train_f1"]:.4f})')


def run_ssl_round(model, train2_df, pseudo_df, epochs, lr, round_num,
                  global_epoch_offset, history_acc):
    """
    Fine-tune the full model on combined supervised + pseudo-labeled data.

    - Supervised data keeps camera augmentations + BASE_TRANSFORM.
    - Pseudo-labeled data uses SSL_TRANSFORM (general aug, no camera transforms).
    - WeightedRandomSampler balances all classes in every batch.
    - Class-weighted CE loss provides additional gradient boost for rare classes.
    """
    print(f'\n{"="*65}')
    print(f'SSL Round {round_num}  |  threshold={SSL_CONF_THRESH[round_num-1]}  '
          f'pseudo={len(pseudo_df):,}  lr={lr}  epochs={epochs}')
    print(f'{"="*65}')

    sup_ds     = PigDatasetCameraAug(train2_df, CAMERA_AUG_FN, BASE_TRANSFORM)
    pseudo_ds  = PseudoLabelDataset(pseudo_df, SSL_TRANSFORM)
    combined   = ConcatDataset([sup_ds, pseudo_ds])

    sup_labels     = [s[2] for s in sup_ds.samples]
    pseudo_labels  = pseudo_df['class_id'].astype(int).tolist()
    all_labels     = sup_labels + pseudo_labels

    loss_weights   = compute_loss_weights(all_labels)
    criterion_ssl  = nn.CrossEntropyLoss(weight=loss_weights)
    loader         = build_weighted_loader(combined, all_labels, BATCH_SIZE)

    combined_counts = np.bincount(all_labels, minlength=NUM_CLASSES)
    print(f'  Supervised (with cam aug): {len(sup_ds):,}')
    print(f'  Pseudo-labeled test:       {len(pseudo_ds):,}')
    print(f'  Combined:                  {len(combined):,}  |  {len(loader)} batches')
    print('  Combined class distribution:')
    for cid, (cnt, w) in enumerate(zip(combined_counts, loss_weights.cpu())):
        print(f'    [{cid}] {CLASS_NAMES[cid]:25s}: {cnt:>6,}  CE-w={w:.3f}')

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_f1, best_state = 0.0, None

    for ep in range(epochs):
        t0 = time.time()
        loss, acc, f1 = train_one_epoch(model, loader, optimizer, criterion_ssl)
        elapsed = time.time() - t0
        global_epoch_offset += 1
        history_acc.append({'stage': f'SSL-R{round_num}', 'epoch': global_epoch_offset,
                             'train_loss': loss, 'train_acc': acc, 'train_f1': f1})
        flag = '  ← best' if f1 > best_f1 else ''
        print(f'  R{round_num} Ep {ep+1:>2}/{epochs} ({elapsed/60:.1f}m)  '
              f'loss={loss:.4f}  acc={acc:.4f}  f1={f1:.4f}{flag}', flush=True)
        if f1 > best_f1:
            best_f1    = f1
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    print(f'  Round {round_num} best macro-F1: {best_f1:.4f}')
    return model, best_f1, global_epoch_offset


print('SSL function defined ✓')

In [ ]:
# ── SSL Round 1  (threshold = 0.95) ──────────────────────────────────────────
print('Generating pseudo-labels (Phase 1 model)...')
pseudo_all_r1 = generate_pseudo_labels(model, test, VAL_TRANSFORM)
pseudo_high_r1 = pseudo_all_r1[pseudo_all_r1['confidence'] >= SSL_CONF_THRESH[0]].copy()

print(f'Threshold {SSL_CONF_THRESH[0]}: '
      f'{len(pseudo_high_r1):,} / {len(pseudo_all_r1):,} '
      f'({100*len(pseudo_high_r1)/len(pseudo_all_r1):.1f}%) kept')
print('Pseudo-label distribution:')
for cid, cnt in sorted(Counter(pseudo_high_r1['class_id'].tolist()).items()):
    print(f'  [{cid}] {CLASS_NAMES[cid]:25s}: {cnt:,}')

# Confidence histogram
plt.figure(figsize=(10, 4))
plt.hist(pseudo_all_r1['confidence'], bins=50, edgecolor='black', alpha=0.75, color='steelblue')
plt.axvline(SSL_CONF_THRESH[0], color='red',    linestyle='--', label=f'R1 thresh={SSL_CONF_THRESH[0]}')
plt.axvline(SSL_CONF_THRESH[1], color='orange', linestyle='--', label=f'R2 thresh={SSL_CONF_THRESH[1]}')
plt.xlabel('Max softmax confidence'); plt.ylabel('Count')
plt.title('Test-set confidence distribution (Phase 1 model)')
plt.legend(); plt.grid(alpha=0.3)
plt.savefig('confidence_r1.png', dpi=120, bbox_inches='tight')
plt.show()

history_ssl = []
global_ep   = len(history_sup)   # offset so SSL epochs continue the x-axis

model, ssl_f1_r1, global_ep = run_ssl_round(
    model, train2, pseudo_high_r1,
    epochs=EPOCHS_SSL, lr=LR_SSL[0],
    round_num=1, global_epoch_offset=global_ep, history_acc=history_ssl
)

In [ ]:
# ── SSL Round 2  (threshold = 0.90, updated model) ───────────────────────────
print('Generating updated pseudo-labels (Round 1 model)...')
pseudo_all_r2  = generate_pseudo_labels(model, test, VAL_TRANSFORM)
pseudo_high_r2 = pseudo_all_r2[pseudo_all_r2['confidence'] >= SSL_CONF_THRESH[1]].copy()

print(f'Threshold {SSL_CONF_THRESH[1]}: '
      f'{len(pseudo_high_r2):,} / {len(pseudo_all_r2):,} '
      f'({100*len(pseudo_high_r2)/len(pseudo_all_r2):.1f}%) kept')

# Confidence shift: Phase 1 → Round 1
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, pall, title in [
    (axes[0], pseudo_all_r1, f'Phase 1 model  (thresh={SSL_CONF_THRESH[0]})'),
    (axes[1], pseudo_all_r2, f'SSL Round 1 model  (thresh={SSL_CONF_THRESH[1]})'),
]:
    ax.hist(pall['confidence'], bins=50, edgecolor='black', alpha=0.75, color='steelblue')
    ax.set_title(title); ax.set_xlabel('Confidence'); ax.grid(alpha=0.3)
plt.suptitle('Confidence distribution shift across SSL rounds', fontweight='bold')
plt.tight_layout()
plt.savefig('confidence_shift.png', dpi=120, bbox_inches='tight')
plt.show()

model, ssl_f1_r2, global_ep = run_ssl_round(
    model, train2, pseudo_high_r2,
    epochs=EPOCHS_SSL, lr=LR_SSL[1],
    round_num=2, global_epoch_offset=global_ep, history_acc=history_ssl
)

In [ ]:
# ── Save final model ──────────────────────────────────────────────────────────
torch.save({
    'model_state_dict': model.state_dict(),
    'class_names':      CLASS_NAMES,
    'phase1_f1':        best_f1_sup,
    'ssl_round1_f1':    ssl_f1_r1,
    'ssl_round2_f1':    ssl_f1_r2,
    'phase':            'ssl_final',
}, SAVE_SSL_FINAL)
print(f'Final model saved → {SAVE_SSL_FINAL}')
print(f'  Phase 1 best F1:  {best_f1_sup:.4f}')
print(f'  SSL Round 1 F1:   {ssl_f1_r1:.4f}')
print(f'  SSL Round 2 F1:   {ssl_f1_r2:.4f}')

from google.colab import files
files.download(SAVE_SSL_FINAL)
print('Model downloaded ✓')

In [ ]:
# ── Full training summary plot ────────────────────────────────────────────────
all_history = history_sup + history_ssl
s1_end = EPOCHS_S1
s2_end = EPOCHS_S1 + EPOCHS_S2
s3_end = EPOCHS_S1 + EPOCHS_S2 + EPOCHS_S3
ssl_start = s3_end
plot_history(
    all_history,
    vlines=[
        (s1_end + 0.5,   'S1→S2'),
        (s2_end + 0.5,   'S2→S3'),
        (ssl_start + 0.5,'S3→SSL'),
        (ssl_start + EPOCHS_SSL + 0.5, 'R1→R2'),
    ],
    title='Full training — EfficientNetV2-S + 3-stage FT + SSL (thresholds 0.95/0.90)',
    fname='full_training_summary.png'
)

## Final Inference & Submission

In [ ]:
print('=' * 65)
print('FINAL INFERENCE')
print('=' * 65)

ckpt = torch.load(SAVE_SSL_FINAL, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded  SSL Round 2 F1={ckpt["ssl_round2_f1"]:.4f}')

test_loader = DataLoader(
    PigTestDataset(test, VAL_TRANSFORM),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

all_row_ids, all_preds = [], []
with torch.no_grad():
    for bi, (imgs, row_ids) in enumerate(test_loader):
        preds = model(imgs.to(DEVICE)).argmax(dim=1).cpu().tolist()
        all_row_ids.extend(list(row_ids))
        all_preds.extend(preds)
        bar = '█' * int(30*(bi+1)/len(test_loader)) + '░' * (30-int(30*(bi+1)/len(test_loader)))
        print(f'\r  |{bar}| {bi+1}/{len(test_loader)}', end='', flush=True)

print(f'\n{len(all_preds):,} predictions')

submission = pd.DataFrame({'row_id': all_row_ids, 'class_id': all_preds})
sample_sub = pd.read_csv(BASE_DIR / 'sample_submission.csv')

assert set(submission['row_id']) == set(sample_sub['row_id']),  'row_id mismatch!'
assert submission['class_id'].between(0, 4).all(),              'class_id out of range!'

submission = (
    submission.set_index('row_id')
              .reindex(sample_sub['row_id'])
              .reset_index()
)

print('\nPrediction distribution:')
for cid, cnt in sorted(submission['class_id'].value_counts().items()):
    print(f'  [{cid}] {CLASS_NAMES[cid]:25s}: {cnt:,}  ({100*cnt/len(submission):.1f}%)')

submission.to_csv(SUBMISSION_PATH, index=False)
print(f'\nSaved → {SUBMISSION_PATH}')
print(submission.head(10).to_string())

from google.colab import files
files.download(SUBMISSION_PATH)
print('Submission downloaded ✓')